# CHB-MIT Seizure Detection — full pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noamshva/Epileptic-Seizure-Classification/blob/main/src/run_pipeline.ipynb)

Runs the whole project end-to-end by driving the `src/` scripts as subprocesses:

1. **Encode** — `1.signal2spikes_chbmit.py` : EEG → SCTN resonator spike trains
2. **Analyze** — `2.analyze_chbmit.py` : SCTN-vs-FFT spectrograms + topography (paper Figs 4/6/7)
3. **Classify** — `3.run_classify_chbmit.py` (SVM) and `3b.run_classify_rf_chbmit.py` (Random Forest)

Scope the run with `PIPELINE_SUBJECTS` in the Config cell (default: `chb01`).
Runs locally or in Colab (see Setup). Training an SNN is out of scope here.

## Setup (Colab only — skip when running locally)

In [ ]:
# Uncomment in Google Colab. Locally, just run the Config cell below.
# 1) Clone the code:
# !git clone https://github.com/noamshva/Epileptic-Seizure-Classification.git
# %cd Epileptic-Seizure-Classification
# 2) Install deps (sctn pulls numba/torch/numpy/... transitively):
# !pip -q install sctn mne scikit-learn
# 3) Mount Drive (your uploaded dataset/ lives here):
# from google.colab import drive; drive.mount('/content/drive')
#
# Then in the Config cell below, set DATA_ROOT to the Drive folder that CONTAINS
# your dataset/ folder, e.g.:
#     DATA_ROOT = "/content/drive/MyDrive/Msc/Epileptic-Seizure-Classification"
# (so the scripts read DATA_ROOT/dataset/chbmit and DATA_ROOT/dataset/spikes_raw)

## Config

In [ ]:
import os, sys, subprocess
from pathlib import Path

# --- where the CODE is (this repo) ---
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir() and (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent          # notebook opened from inside src/
os.chdir(REPO_ROOT)
PY = sys.executable

# --- where the DATA lives (folder that CONTAINS dataset/chbmit + dataset/spikes_raw) ---
# Local: leave as REPO_ROOT.  Colab: point at your Drive folder, e.g.
#   DATA_ROOT = "/content/drive/MyDrive/Msc/Epileptic-Seizure-Classification"
DATA_ROOT = str(REPO_ROOT)
os.environ["PIPELINE_DATA_ROOT"] = DATA_ROOT     # scripts read data + write outputs under here
DATA = Path(DATA_ROOT)

# --- scope ---
# Analysis (Stage 2) = ONE subject (like the paper); encode (1) + classify (3) = ALL patients.
ALL_PATIENTS = sorted(p.name for p in (DATA / "dataset" / "spikes_raw").glob("chb*")
                      if p.is_dir()) or ["chb01"]
os.environ["PIPELINE_SUBJECTS"]      = ",".join(ALL_PATIENTS)
os.environ["PIPELINE_SUBJECT"]       = "chb01"
os.environ["PIPELINE_EDF"]           = "chb01_03.edf"
os.environ["PIPELINE_SEIZURE_MIN"]   = "49"
os.environ["PIPELINE_NONSEIZURE_MIN"]= "4"
CLASSIFY_PATIENTS = ALL_PATIENTS

print("Repo (code):", REPO_ROOT)
print("Data root  :", DATA_ROOT)
print("Python     :", PY)
print("Patients   :", ALL_PATIENTS)
have_edf    = (DATA / "dataset" / "chbmit").is_dir()
have_spikes = (DATA / "dataset" / "spikes_raw").is_dir()
print("dataset/chbmit:", have_edf, " | dataset/spikes_raw:", have_spikes)
assert have_edf or have_spikes, "No dataset under DATA_ROOT — set DATA_ROOT to your data folder."


def run_stage(script_args, desc, tail=6000):
    print(f"\n{'='*70}\n{desc}\n$ {PY} {' '.join(script_args)}\n{'='*70}", flush=True)
    r = subprocess.run([PY, "-u", *script_args], env=os.environ, text=True,
                       capture_output=True, errors="replace")
    out = r.stdout or ""
    print(out[-tail:])
    if r.returncode != 0:
        print("---- STDERR ----\n", (r.stderr or "")[-3000:])
        raise RuntimeError(f"{desc} failed (exit {r.returncode})")
    return out


def show_figs(*names):
    try:
        from IPython.display import Image, display
    except Exception:
        return
    for n in names:
        p = DATA / "analysis" / "figures" / n
        if p.is_file():
            print(n); display(Image(filename=str(p)))

def show_df(df):
    try:
        from IPython.display import display; display(df)
    except Exception:
        print(df.to_string())

## Stage 1 — Encode (EEG → SCTN spikes)  ·  all patients

Runs the encoder on every patient in `PIPELINE_SUBJECTS`. Already-encoded
recordings are **skipped**, so if you've already encoded everything this stage
is just a fast verification (seconds–minutes). A *fresh* full encode is hours —
if the spikes are already present you can skip this cell.

In [ ]:
run_stage(["src/1.signal2spikes_chbmit.py"], "STAGE 1 — ENCODE")

## Stage 2 — Analyze (SCTN vs FFT)  ·  one illustrative subject

Reproduces the Situational-Awareness paper's analysis figures (Figs 4/6/7) for a
**single example recording** (`PIPELINE_SUBJECT` / `PIPELINE_EDF`, default
chb01_03) — like the paper, which shows one example. FFT-vs-SCTN spectrograms,
per-channel grids, and topography, written to `analysis/figures/`.

In [ ]:
run_stage(["src/2.analyze_chbmit.py"], "STAGE 2 — ANALYZE")
show_figs("04_fft_vs_sctn_bands.png",
          "08_topography_fft_vs_sctn.png",
          "09_topography_seizure_vs_non_seizure.png")

## Stage 3 — Classify (SVM + Random Forest)  ·  all patients

Trains the classical classifiers on the SCTN burstiness features for **every
patient in `CLASSIFY_PATIENTS`** and reports per-patient honest
(leave-one-recording-out) AUC/F1 + a pooled confusion matrix. Results go to
`output_data/results/`. Running all 24 patients rebuilds features per patient,
so this takes a while (tens of minutes) — reduce `CLASSIFY_PATIENTS` in Config
for a quick pass.

In [ ]:
run_stage(["src/3.run_classify_chbmit.py",  *CLASSIFY_PATIENTS], "STAGE 3a — SVM")
run_stage(["src/3b.run_classify_rf_chbmit.py", *CLASSIFY_PATIENTS], "STAGE 3b — Random Forest")

import pandas as pd
res = DATA / "output_data" / "results"
for name in ("svm_classifier_report.csv", "rf_classifier_report.csv"):
    p = res / name
    if p.is_file():
        print("\n" + name); show_df(pd.read_csv(p).round(3))
try:
    from IPython.display import Image, display
    for name in ("svm_confusion_matrix.png", "rf_confusion_matrix.png"):
        p = res / name
        if p.is_file():
            display(Image(filename=str(p)))
except Exception:
    pass

## Done

All under **`DATA_ROOT`** (your Drive folder in Colab, the repo locally):
- Spikes  → `dataset/spikes_raw/`
- Figures → `analysis/figures/`  ·  stats → `analysis/stats/`
- Classifier reports → `output_data/results/`

To change scope: edit `CLASSIFY_PATIENTS` / `PIPELINE_SUBJECT` in Config and re-run.
For the SNN classifier, see the SNN branch (out of scope for the classical pipeline).